<a href="https://colab.research.google.com/github/talgiladi/Portfolio/blob/main/pytorch_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [13]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root = "./data", train = True, transform = transform, download = True)
trainloader = torch.utils.data.DataLoader(trainset, batch_size = 4, shuffle = True, num_workers = 2)

testset = torchvision.datasets.CIFAR10(root = "./data", train = False, download = True, transform = transform)
testloader = torch.utils.data.DataLoader(testset, batch_size = 4, shuffle = False, num_workers = 2)

In [12]:
#next(iter(trainset))[0][0].shape

torch.Size([32, 32])

In [22]:
import torch.nn as nn
import torch.nn.functional as F
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(3, 6, 5)  #->  32 - 5 + 1 = 28. after max pool -> 14
    self.conv2 = nn.Conv2d(6, 16, 5) #-> 14 - 5 + 1 = 10. after max pool -> 5
    self.max_pool = nn.MaxPool2d(2, 2);
    self.fc1 = nn.Linear(5 * 5 * 16, 120) #->5 * 5 (image size) * 16 (channels)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x) -> torch.Tensor:
    x = self.max_pool(F.relu(self.conv1(x)))
    x = self.max_pool(F.relu(self.conv2(x)))
    x = x.view(-1, 5* 5 * 16)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

net = Net()

In [23]:
optimizer = torch.optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
  net.train()
  running_loss = 0.0
  for i, data in enumerate(trainloader, 0):
    input, labels = data
    optimizer.zero_grad()
    output = net(input)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

    if (i%1000 == 999):
      print(f"Epoch {epoch + 1}, batch {i + 1}, loss {running_loss/1000}")
      running_loss = 0.0

print("Finished training")

Epoch 1, batch 1000, loss 2.2864492865800856
Epoch 1, batch 2000, loss 2.0954949185848237
Epoch 1, batch 3000, loss 1.8978191897273065
Epoch 1, batch 4000, loss 1.7699813376665114
Epoch 1, batch 5000, loss 1.6895450419783593
Epoch 1, batch 6000, loss 1.6065707860291005
Epoch 1, batch 7000, loss 1.586530130982399
Epoch 1, batch 8000, loss 1.5242939855456352
Epoch 1, batch 9000, loss 1.5234943119287492
Epoch 1, batch 10000, loss 1.4983944561779499
Epoch 1, batch 11000, loss 1.408124858736992
Epoch 1, batch 12000, loss 1.431192408502102
Epoch 2, batch 1000, loss 1.401728873848915
Epoch 2, batch 2000, loss 1.3764239778220653
Epoch 2, batch 3000, loss 1.335312748476863
Epoch 2, batch 4000, loss 1.3427479067593813
Epoch 2, batch 5000, loss 1.3384344148933887
Epoch 2, batch 6000, loss 1.3258334194272756
Epoch 2, batch 7000, loss 1.309853298522532
Epoch 2, batch 8000, loss 1.3183593651652337
Epoch 2, batch 9000, loss 1.2787926704883577
Epoch 2, batch 10000, loss 1.2849598798006774
Epoch 2, bat

In [24]:
path = "/content/drive/MyDrive/Colab Notebooks/Models/cifar10_0.pth"
torch.save(obj = net.state_dict(), f = path)

In [29]:
net.eval()
correct = 0
total = 0
with torch.inference_mode():
  for data in testloader:
    input, labels = data
    output = net(input)
    total += labels.size(0)
    _, predicted = torch.max(output.data, 1)
    correct += torch.eq(predicted , labels).sum().item()

accuracy = 100 * (correct / total)
print(f"Test accuracy {accuracy}")

Test accuracy 63.72
